<a href="https://colab.research.google.com/github/kosaquito/ProcesamientoDelHabla/blob/main/PH_taller_intro_pdfs_2026_09_07_parte_1_Martinez_Martin.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Taller: Manipulación de archivos PDF con Python 🐍📄 en Programación para ciencia de datos

## Introducción al procesamiento de PDFs con `pdfplumber` y `PyMuPDF`

En este taller vas a aprender a **abrir, inspeccionar, modificar y extraer información** de archivos PDF usando dos librerías muy usadas en Python:

- **[`pdfplumber`](https://github.com/jsvine/pdfplumber)**: excelente para extraer texto y **tablas** de forma sencilla y muy legible.
- **[`PyMuPDF`](https://pymupdf.readthedocs.io/) (se importa como `pymupdf` o, de forma más antigua, `fitz`)**: muy rápida, potente para extraer texto, imágenes y **modificar metadata**.




### Objetivos del taller

Al finalizar este notebook vas a poder:

1. Instalar y cargar las librerías necesarias.
2. Descargar un PDF público desde internet.
3. Importar/abrir un PDF con ambas librerías.
4. Obtener la **metadata** del documento.
5. **Modificar** esa metadata y guardar un nuevo PDF.
6. Acceder a una **página específica** del documento.
7. Extraer **todo el texto** del PDF.
8. Extraer una **tabla** y convertirla en un `DataFrame` de `pandas`.

> ⚠️ **Fuera de alcance:** en este taller **no** trabajaremos con PDFs escaneados (imágenes de texto) ni con OCR. Todos los PDFs que usaremos tienen texto "real" embebido, extraíble directamente.

### 📄 El documento de trabajo

Vamos a usar un documento **público y real**: el **WARN Report** del Estado de California (EDD - Employment Development Department), un reporte oficial que lista avisos de despidos masivos y cierres de empresas. Es un PDF generado desde Excel, con texto y tablas reales (no escaneado), ideal para practicar.

A lo largo del notebook vas a encontrar:

- 🧩 **Actividad**: código que tenés que ejecutar y observar.
- 🤔 **Pregunta**: para que reflexiones o compares salidas antes de seguir.
- 🚀 **Desafío**: ejercicios para que resuelvas vos con lo aprendido.

¡Empecemos!

---
## 1. Instalación y carga de librerías

Vamos a instalar las tres librerías principales que usaremos:

- `pdfplumber`
- `pymupdf` (PyMuPDF)
- `pandas` (ya la conocés, para trabajar con tablas)
- `requests` (para descargar el PDF desde internet)

Si estás en **Google Colab** o en tu propio entorno, ejecutá la siguiente celda. El símbolo `!` al principio le indica al notebook que ejecute un comando de la terminal en lugar de código Python.


In [ ]:
!pip install pdfplumber pymupdf pandas requests -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 89.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 124.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 112.4 MB/s eta 0:00:00


Ahora cargamos las librerías en nuestro entorno de trabajo.

💡 **Nota:** `PyMuPDF` se instala con el nombre `pymupdf`, pero durante muchos años se importaba como `fitz` (su nombre histórico). Ambas formas siguen funcionando, aunque la documentación oficial recomienda usar `import pymupdf`.


In [ ]:
import pdfplumber
import pymupdf          # PyMuPDF. Forma equivalente y muy usada: "import fitz"
import pandas as pd
import requests
import os

print("pdfplumber:", pdfplumber.__version__)
print("pymupdf:", pymupdf.__doc__.splitlines()[0] if pymupdf.__doc__ else pymupdf.version)
print("pandas:", pd.__version__)

pdfplumber: 0.11.10
pymupdf: PyMuPDF 1.28.2: Python bindings for the MuPDF 1.28.2 library.
pandas: 2.2.3


🤔 **Pregunta:** ¿Qué versión de cada librería te imprimió la celda anterior? Anotala: te puede servir más adelante si algo no funciona como se describe en este notebook (las librerías cambian de versión en versión).

**Respuesta (Martin):** La celda anterior imprimió `pdfplumber: 0.11.10`, `pymupdf: PyMuPDF 1.28.2` y `pandas: 2.2.3`. Fundamental tomar nota porque si en el futuro algo de este notebook no funciona igual, lo primero a revisar es si cambió la versión de alguna de estas librerías (sobre todo `pdfplumber`, que actualiza seguido su lógica de detección de tablas).

---
## 2. Descargar el PDF de ejemplo

Vamos a descargar el **WARN Report de California** directamente desde un repositorio público de GitHub (es el mismo archivo que usa la documentación oficial de `pdfplumber` como ejemplo).

🧩 **Actividad:** ejecutá la siguiente celda. Va a descargar el PDF y guardarlo en el disco con el nombre `warn_report.pdf`.


In [ ]:
URL_PDF = "https://raw.githubusercontent.com/jsvine/pdfplumber/stable/examples/pdfs/ca-warn-report.pdf"
NOMBRE_ARCHIVO = "warn_report.pdf"

respuesta = requests.get(URL_PDF)
print("Código de estado HTTP:", respuesta.status_code)

with open(NOMBRE_ARCHIVO, "wb") as f:
    f.write(respuesta.content)

tamano_kb = os.path.getsize(NOMBRE_ARCHIVO) / 1024
print(f"Archivo guardado como '{NOMBRE_ARCHIVO}' ({tamano_kb:.1f} KB)")

Código de estado HTTP: 200
Archivo guardado como 'warn_report.pdf' (467.4 KB)


🤔 **Preguntas:**

1. ¿Qué código de estado HTTP obtuviste? ¿Qué código esperarías ver si la descarga fue exitosa?



**Tu respuesta (Martin):** El código HTTP obtenido fue `200`, que es lo que se espera cuando la descarga es exitosa: `200 OK` significa que el servidor encontró el recurso y lo devolvió sin errores.

2. ¿Qué pasaría si la URL estuviera mal escrita o el archivo ya no existiera? Probá cambiar temporalmente la URL por una inventada (por ejemplo, agregale `x` al final) y volvé a ejecutar la celda. ¿Qué valor toma `respuesta.status_code`? Después no te olvides de volver a poner la URL correcta y ejecutar la celda de nuevo antes de seguir.

**Tu respuesta (Martin):** Si a la URL le agrego una `x` al final (o está mal escrita / el archivo ya no existe), `requests.get()` no lanza ninguna excepción: simplemente devuelve una respuesta con `status_code = 404` (Not Found), y `respuesta.content` contiene el HTML de la página de error de GitHub en lugar del PDF. Si no se valida `status_code` antes de escribir el archivo, terminamos guardando ese HTML de error con extensión `.pdf`, lo cual rompe todo el resto del notebook al intentar abrirlo. Por eso siempre conviene chequear el código de estado, tal cual lo explica en la clase, antes de asumir que la descarga funcionó.

---
## 3. Importar el PDF

Ahora que tenemos el archivo en disco, vamos a **abrirlo** con las dos librerías y comparar cómo se comporta cada una.

### 3.1 Con `pdfplumber`


In [ ]:
pdf_plumber = pdfplumber.open(NOMBRE_ARCHIVO)

print("Tipo de objeto:", type(pdf_plumber))
print("Cantidad de páginas:", len(pdf_plumber.pages))

Tipo de objeto: <class 'pdfplumber.pdf.PDF'>
Cantidad de páginas: 16


### 3.2 Con `PyMuPDF`

In [ ]:
doc = pymupdf.open(NOMBRE_ARCHIVO)

print("Tipo de objeto:", type(doc))
print("Cantidad de páginas:", doc.page_count)

Tipo de objeto: <class 'pymupdf.Document'>
Cantidad de páginas: 16


🤔 **Preguntas:**

1. ¿Coincide la cantidad de páginas reportada por ambas librerías?



**Respuesta (Martin):** Sí, ambas coinciden: tanto `pdfplumber` como `PyMuPDF` reportan 16 páginas para el mismo archivo. Tiene sentido, porque la cantidad de páginas es una propiedad objetiva de la estructura del PDF y no depende de qué librería se use para leerlo.

2. Los objetos `pdf_plumber` y `doc` son de tipos distintos (`pdfplumber.pdf.PDF` vs `pymupdf.Document`). ¿Qué creés que significa esto en términos de cómo cada librería representa internamente un PDF?


**Respuesta (Martin):** Significa que cada librería envuelve el PDF con su propio modelo interno de objetos, aunque ambas representan el mismo estándar de archivo. `pdfplumber` está construido sobre `pdfminer.six` y expone una capa de alto nivel pensada para analizar el *layout* (texto, líneas, tablas) de forma legible en Python puro. `PyMuPDF` envuelve la librería en C `MuPDF` y expone su propio wrapper (`pymupdf.Document`), orientado a performance y a operaciones de bajo nivel (render, edición, metadata). No existe un único objeto 'universal' para un PDF: cada librería decide cómo modelarlo según para qué está pensada.

3. Fijate que abrimos el **mismo archivo** dos veces, con dos variables distintas (`pdf_plumber` y `doc`). Vamos a mantener ambos objetos abiertos durante todo el notebook para poder comparar sus resultados paso a paso.

**Respuesta (Martin):** Con `pdf_plumber` y `doc` abiertos en simultáneo durante todo el notebook podremos comparar, sección por sección, cómo responde cada librería sobre el mismo documento.

---
## 4. Obtener la metadata del documento

La **metadata** de un PDF es información *sobre* el documento (no es el contenido en sí): autor, fecha de creación, título, software con el que fue generado, etc. Se guarda en una sección especial del archivo.

### 4.1 Metadata con `pdfplumber`


In [ ]:
metadata_plumber = pdf_plumber.metadata
metadata_plumber

{'Author': 'Cuellar-Lopez, Monica',
 'CreationDate': "D:20160325082400-07'00'",
 'ModDate': "D:20160325082400-07'00'",
 'Producer': 'Microsoft® Excel® 2013',
 'Creator': 'Microsoft® Excel® 2013'}

### 4.2 Metadata con `PyMuPDF`

In [ ]:
metadata_mupdf = doc.metadata
metadata_mupdf

{'format': 'PDF 1.5',
 'title': '',
 'author': 'Cuellar-Lopez, Monica',
 'subject': '',
 'keywords': '',
 'creator': 'Microsoft® Excel® 2013',
 'producer': 'Microsoft® Excel® 2013',
 'creationDate': "D:20160325082400-07'00'",
 'modDate': "D:20160325082400-07'00'",
 'trapped': '',
 'encryption': None}

🤔 **Preguntas:**

1. ¿Qué campos (claves) aparecen en un diccionario de metadata y no en el otro?



**Respuesta (Martin):** El diccionario de `pdfplumber` (`metadata_plumber`) solo incluye las claves que efectivamente tienen un valor en el PDF: `Author`, `CreationDate`, `ModDate`, `Producer` y `Creator`. El de `PyMuPDF` (`metadata_mupdf`) siempre devuelve un conjunto fijo de claves —`format`, `title`, `author`, `subject`, `keywords`, `creator`, `producer`, `creationDate`, `modDate`, `trapped`, `encryption`— completando con string vacío (`''`) o `None` las que no están presentes en el archivo. Por eso `format`, `title`, `subject`, `keywords`, `trapped` y `encryption` aparecen únicamente en la versión de PyMuPDF.

2. ¿En qué formato está escrita la fecha de creación (`CreationDate` / `creationDate`)? Ese formato se llama **formato de fecha PDF** (`D:AAAAMMDDHHmmSS+ZZ'zz'`). ¿Podrías escribir en una celda de código cómo convertirías ese string a un objeto `datetime` de Python? (Pista: buscá el módulo `datetime.strptime`, vas a tener que armar el formato a mano ya que no es un formato estándar de ISO).


**Respuesta (Martin):** Está en el formato propio de fecha de PDF: `D:AAAAMMDDHHmmSS+ZZ'zz'` (en este caso `"D:20160325082400-07'00'"`). Para convertirlo a un `datetime` de Python hay que sacar el prefijo `"D:"`, reemplazar las comillas simples por `":"` y armar un formato a medida para `strptime` (no es ISO estándar, así que no lo interpreta solo):

```python
from datetime import datetime

fecha_str = metadata_mupdf["creationDate"]     # "D:20160325082400-07'00'"
fecha_limpia = fecha_str.replace("D:", "").replace("'", ":")
if fecha_limpia.endswith(":"):
    fecha_limpia = fecha_limpia[:-1]

fecha_dt = datetime.strptime(fecha_limpia, "%Y%m%d%H%M%S%z")
print(fecha_dt)   # 2016-03-25 08:24:00-07:00
```

Lo probé (Martin) y funciona: devuelve `2016-03-25 08:24:00-07:00`.

3. ¿Quién es el autor del documento? ¿Con qué programa fue generado el PDF originalmente?

**Respuesta (Martin):** El autor es `Cuellar-Lopez, Monica`, y el documento fue generado originalmente con `Microsoft® Excel® 2013` (así figura tanto en `Creator` como en `Producer`), lo cual confirma que el PDF salió de exportar una planilla de cálculo.

---
## 5. Modificar la metadata

`pdfplumber` está pensado **solo para lectura**: no ofrece una forma de modificar y volver a guardar un PDF. Para esto vamos a usar `PyMuPDF`, que sí permite escribir metadata nueva y guardar el resultado en un archivo distinto.

🧩 **Actividad:** vamos a reemplazar la metadata original por una nueva, indicando que este archivo fue procesado por vos en este taller.


In [ ]:
nueva_metadata = {
    "title": "WARN Report de California - Procesado en taller de PDFs",
    "author": "Escribí tu nombre acá",
    "subject": "Avisos de despidos masivos y cierres de empresas (California EDD)",
    "keywords": "WARN, despidos, California, taller-python, pdfplumber, pymupdf",
    "creator": "Taller de manipulación de PDFs con Python",
}

doc.set_metadata(nueva_metadata)
doc.save("warn_report_modificado.pdf")

print("Nuevo archivo guardado: warn_report_modificado.pdf")

Nuevo archivo guardado: warn_report_modificado.pdf


Para confirmar que el cambio se aplicó correctamente, vamos a **abrir el archivo nuevo** (no el objeto `doc`, que ya tiene la metadata en memoria, sino el PDF tal como quedó guardado en disco) y revisar su metadata.


In [ ]:
doc_modificado = pymupdf.open("warn_report_modificado.pdf")
doc_modificado.metadata

{'format': 'PDF 1.5',
 'title': 'WARN Report de California - Procesado en taller de PDFs',
 'author': 'Escribí tu nombre acá',
 'subject': 'Avisos de despidos masivos y cierres de empresas (California EDD)',
 'keywords': 'WARN, despidos, California, taller-python, pdfplumber, pymupdf',
 'creator': 'Taller de manipulación de PDFs con Python',
 'producer': 'Microsoft® Excel® 2013',
 'creationDate': "D:20160325082400-07'00'",
 'modDate': "D:20160325082400-07'00'",
 'trapped': '',
 'encryption': None}

🤔 **Preguntas:**

1. Compará `metadata_mupdf` (la metadata original, de la sección 4.2) con la metadata de `doc_modificado`. ¿Qué campos cambiaron? ¿Qué campos se mantuvieron igual (por ejemplo, `producer` o las fechas)?



**Respuesta (Martin):** Cambiaron `title`, `author`, `subject`, `keywords` y `creator` (los campos que reemplazamos explícitamente en `nueva_metadata`). Se mantuvieron iguales `producer` (sigue diciendo `Microsoft® Excel® 2013`, porque no lo tocamos), `creationDate` y `modDate` (las fechas originales no cambian solo por reescribir metadata de texto), además de `format`, `trapped` y `encryption`.

2. ¿Por qué creés que guardamos el resultado en un archivo **nuevo** (`warn_report_modificado.pdf`) en lugar de sobrescribir `warn_report.pdf`? Pensá qué pasaría si necesitáramos volver a comparar contra el original más adelante.

**Respuesta (Martin):** Guardarlo en un archivo nuevo preserva el original intacto en disco para poder seguir comparando contra él más adelante (por ejemplo, en la próxima celda comparamos `metadata_mupdf` original contra la metadata de `doc_modificado`). Si hubiéramos sobrescrito `warn_report.pdf`, perderíamos esa referencia y no podríamos volver a analizar cómo era antes del cambio; además, sobrescribir el archivo que `doc` tiene abierto en ese momento puede fallar o corromper el archivo si no se usa el modo incremental.


3. `doc.save(...)` puede fallar si intentás guardar sobre el **mismo archivo que tenés abierto** sin usar el modo incremental. Investigá brevemente qué es el parámetro `incremental=True` de `doc.save()` y en qué casos se usa.

**Respuesta (Martin):** `incremental=True` le indica a PyMuPDF que, en lugar de reescribir el PDF completo desde cero, agregue los cambios como una nueva sección al final del archivo existente (una actualización incremental, tal como lo define el estándar PDF). Se usa típicamente cuando se quiere guardar sobre el **mismo archivo que ya está abierto** (por ejemplo, tras agregar una anotación o modificar metadata), y es más rápido porque no reprocesa todo el documento. Tiene restricciones: no se puede combinar con `garbage collection` (`garbage=...`) ni usarse sobre un documento que fue creado en memoria (no abierto desde un archivo existente).

---
## 6. Acceder a una página específica

Los PDFs se representan internamente como una **lista de páginas**. En Python, eso significa que podemos acceder a una página puntual usando su índice, **empezando en 0** (la primera página es la página `0`).

Vamos a trabajar con la **página 6** del documento (índice `5`).

### 6.1 Con `pdfplumber`


In [ ]:
numero_pagina = 5  # página 6 "humana" = índice 5

pagina_plumber = pdf_plumber.pages[numero_pagina]

print("Número de página (pdfplumber):", pagina_plumber.page_number)
print("Ancho:", pagina_plumber.width, "- Alto:", pagina_plumber.height)

Número de página (pdfplumber): 6
Ancho: 792 - Alto: 612


### 6.2 Con `PyMuPDF`

In [ ]:
pagina_mupdf = doc[numero_pagina]

print("Número de página (índice, PyMuPDF):", pagina_mupdf.number)
rect = pagina_mupdf.rect
print("Ancho:", rect.width, "- Alto:", rect.height)

Número de página (índice, PyMuPDF): 5
Ancho: 792.0 - Alto: 612.0


Ahora extraigamos el **texto** de esa página específica con cada librería y comparemos:

In [ ]:
texto_pagina_plumber = pagina_plumber.extract_text()
print("----- pdfplumber -----")
print(texto_pagina_plumber[:500])

----- pdfplumber -----
09/18/2015 1 1 / 2 0 / 2 0 15 0 9 / 2 8 / 2 0 15 Boeing Company Huntington Beach 23 Layoff Permanent
09/21/2015 1 1 / 2 0 / 2 0 15 0 9 / 2 8 / 2 0 15 Safeway Inc. Pleasanton 15 Layoff Unknown at this time
09/24/2015 1 1 / 2 7 / 2 0 15 0 9 / 2 8 / 2 0 15 Brice Manufacturing Company, Inc. Pacoima 9 Layoff Permanent
09/28/2015 1 1 / 2 7 / 2 0 15 0 9 / 2 8 / 2 0 15 Safeway, Inc. Pleasanton 4 Layoff Unknown at this time
09/24/2015 0 9 / 2 4 / 2 0 15 0 9 / 2 9 / 2 0 15 Space Systems/Loral, LLC Palo Al


In [ ]:
texto_pagina_mupdf = pagina_mupdf.get_text()
print("----- PyMuPDF -----")
print(texto_pagina_mupdf[:500])

----- PyMuPDF -----
09/18/2015     
09/21/2015     
09/24/2015     
09/28/2015     
09/24/2015     
09/25/2015     
09/30/2015     
09/30/2015     
09/30/2015     
09/28/2015     
09/30/2015     
09/30/2015     
09/30/2015     
09/24/2015     
10/01/2015     
10/02/2015     
09/29/2015     
10/02/2015     
10/02/2015     
10/05/2015     
10/05/2015     
10/06/2015     
10/06/2015     
10/08/2015     
10/06/2015     
10/06/2015     
10/07/2015     
10/06/2015     
10/07/2015     
10/08/2015     
10/08/2015     
10/0


🤔 **Preguntas:**

1. Observá con atención ambas salidas. ¿Se ve igual el texto extraído por `pdfplumber` que el extraído por `PyMuPDF`? ¿Qué diferencias notás en cómo se ordenan o separan los datos de las columnas de la tabla?



**Respuesta (Martin):** No, no se ve igual. La salida de `pdfplumber` agrupa cada fila de la tabla en una sola línea de texto legible (fecha, empresa, ciudad, etc. todo junto), porque su algoritmo agrupa los caracteres por posición vertical (misma línea visual) y horizontal antes de imprimir. La salida de `PyMuPDF` con `get_text()` en modo simple, en cambio, respeta la estructura interna de líneas de texto del PDF tal cual está codificada: como en este documento cada dígito de las fechas fue puesto como un elemento de texto separado, terminan apareciendo en líneas sueltas casi vacías, perdiendo la estructura de columnas de la tabla.

2. `pdfplumber.pages[numero_pagina].page_number` devuelve un número, mientras que `doc[numero_pagina].number` devuelve otro. ¿Qué diferencia hay entre ambos? (Pista: uno arranca en 0 y el otro no).


**Respuesta (Martin):** `pdfplumber` numera las páginas empezando en 1 (`page_number` de la página de índice 5 es `6`), mientras que `PyMuPDF` usa directamente el índice interno, que arranca en 0 (`.number` es `5`). Es decir, uno es el número de página "humano" y el otro es el índice de la lista interna.

3. Cambiá el valor de `numero_pagina` por otro (por ejemplo, `0` o `10`) y volvé a ejecutar las celdas de esta sección. ¿El contenido cambia como esperabas?

**Respuesta (Martin):** Sí, el contenido cambia como se esperaba: al cambiar `numero_pagina` (por ejemplo a `0` o `10`), tanto `pagina_plumber` como `pagina_mupdf` pasan a apuntar a esa nueva página, y el texto extraído en las celdas siguientes corresponde a esa página distinta (otro rango de fechas y empresas del reporte).

---
# Parte 2

## 7. Obtener todo el texto del documento

Para extraer el texto completo, tenemos que **recorrer todas las páginas** y concatenar el resultado de cada una.

### 7.1 Con `pdfplumber`


In [ ]:
texto_completo_plumber = ""

for pagina in pdf_plumber.pages:
    texto_pagina = pagina.extract_text()
    if texto_pagina:               # algunas páginas podrían no tener texto
        texto_completo_plumber += texto_pagina + "\n"

print("Cantidad de caracteres extraídos (pdfplumber):", len(texto_completo_plumber))
print("Cantidad de palabras aproximada:", len(texto_completo_plumber.split()))

Cantidad de caracteres extraídos (pdfplumber): 66803
Cantidad de palabras aproximada: 17368


### 7.2 Con `PyMuPDF`

`PyMuPDF` tiene un atajo, `doc.get_text()` sobre el propio documento, pero para practicar el recorrido de páginas lo vamos a hacer explícitamente igual que con `pdfplumber`.


In [ ]:
texto_completo_mupdf = ""

for pagina in doc:
    texto_pagina = pagina.get_text()
    if texto_pagina:
        texto_completo_mupdf += texto_pagina + "\n"

print("Cantidad de caracteres extraídos (PyMuPDF):", len(texto_completo_mupdf))
print("Cantidad de palabras aproximada:", len(texto_completo_mupdf.split()))

Cantidad de caracteres extraídos (PyMuPDF): 62438
Cantidad de palabras aproximada: 7240


🤔 **Preguntas:**

1. ¿La cantidad de caracteres/palabras extraídas es idéntica entre ambas librerías? Si no lo es, ¿a qué creés que se debe la diferencia?



**Respuesta (Martin):** No es idéntica: `pdfplumber` extrajo 66803 caracteres y aproximadamente 17368 "palabras", mientras que `PyMuPDF` extrajo 62438 caracteres y solo 7240 "palabras". La diferencia de caracteres se debe a que cada librería agrupa y separa el texto con criterios distintos (espacios extra, saltos de línea, orden de lectura). La diferencia mucho mayor en el conteo de "palabras" se explica porque `texto.split()` cuenta tokens separados por espacios: como en este PDF cada dígito de las fechas está como carácter individual, `pdfplumber` (que junta todo en una sola línea por fila) termina generando muchos más tokens separados por espacio que `PyMuPDF` (que separa esos mismos dígitos en líneas propias, contando menos "palabras" por línea).

2. En el `for pagina in pdf_plumber.pages:` iteramos sobre `.pages`, mientras que en `for pagina in doc:` iteramos directamente sobre `doc`. ¿Qué te dice esto sobre cómo cada librería diseñó su API? (Ambos objetos son *iterables* de páginas, pero se accede de forma distinta).

**Respuesta (Martin):** Dice que ambas librerías diseñaron su API de forma distinta aunque el resultado conceptual sea el mismo (iterar las páginas). `pdfplumber` expone las páginas como un atributo explícito (`.pages`), dejando claro que es una colección dentro del objeto PDF. `PyMuPDF` hizo que el propio objeto `Document` sea iterable (implementa `__iter__`), permitiendo escribir `for pagina in doc` directamente, una API más compacta pero un poco menos explícita sobre qué se está recorriendo.


3. 🚀 **Desafío:** usando `texto_completo_plumber`, contá cuántas veces aparece la palabra `"Layoff"` (despido) en todo el documento. Pista: podés usar el método `.count()` de los strings de Python.

**Respuesta (Martin):** La palabra "Layoff" aparece 400 veces en `texto_completo_plumber` (lo calculé con `.count("Layoff")` en la celda de código siguiente).

In [ ]:
#  Tu turno: contá cuántas veces aparece "Layoff" en texto_completo_plumber

# Martin: uso el método .count() de los strings, tal como sugiere la pista del enunciado.
cantidad_layoff = texto_completo_plumber.count("Layoff")
print("Martin - Cantidad de veces que aparece 'Layoff':", cantidad_layoff)

Martin - Cantidad de veces que aparece 'Layoff': 400


---
## 8. Obtener una tabla y guardarla en `pandas`

Esta es una de las funcionalidades más potentes de `pdfplumber`: el método `.extract_table()` (o `.extract_tables()` si hay varias tablas en la página), que detecta automáticamente las líneas/espacios de una tabla y la devuelve como una **lista de listas**.

Vamos a extraer la tabla de la **primera página** (índice `0`), que contiene el encabezado de columnas.


In [ ]:
primera_pagina = pdf_plumber.pages[0]
tabla_cruda = primera_pagina.extract_table()

print("Tipo de dato:", type(tabla_cruda))
print("Cantidad de filas:", len(tabla_cruda))
print("Primeras 3 filas:")
for fila in tabla_cruda[:3]:
    print(fila)

Tipo de dato: <class 'list'>
Cantidad de filas: 37
Primeras 3 filas:
['Notice Date', 'Effective', 'Received', 'Company', 'City', 'No. Of', 'Layoff/Closure']
['06/22/2015', '0 3 / 2 5 / 2 0 16', '0 7 / 0 1 / 2 0 15', 'Maxim Integrated Product', 'San Jose', '150', 'Closure Permanent']
['06/30/2015', '0 8 / 2 9 / 2 0 15', '0 7 / 0 1 / 2 0 15', 'McGraw-Hill Education', 'Monterey', '137', 'Layoff Unknown at this time']


🤔 **Pregunta:** ¿Qué tipo de estructura de datos es `tabla_cruda`? ¿Y cada elemento dentro de ella (cada fila)? Antes de seguir, fijate cuál es la fila que contiene los **nombres de las columnas**.

Ahora convirtamos esa lista de listas en un `DataFrame` de `pandas`, usando la primera fila como encabezado de columnas:


In [ ]:
encabezados = tabla_cruda[0]
filas_datos = tabla_cruda[1:]

df = pd.DataFrame(filas_datos, columns=encabezados)
df.head()

,Notice Date,Effective,Received,Company,City,No. Of,Layoff/Closure
0,06/22/2015,0 3 / 2 5 / 2 0 16,0 7 / 0 1 / 2 0 15,Maxim Integrated Product,San Jose,150,Closure Permanent
1,06/30/2015,0 8 / 2 9 / 2 0 15,0 7 / 0 1 / 2 0 15,McGraw-Hill Education,Monterey,137,Layoff Unknown at this time
2,06/30/2015,0 8 / 3 0 / 2 0 15,0 7 / 0 1 / 2 0 15,Long Beach Memorial Medical Center,Long Beach,90,Layoff Permanent
3,07/01/2015,0 9 / 0 2 / 2 0 15,0 7 / 0 1 / 2 0 15,Leidos,El Segundo,72,Layoff Permanent
4,07/01/2015,0 9 / 3 0 / 2 0 16,0 7 / 0 1 / 2 0 15,"Bosch Healthcare Systems, Inc.",Palo Alto,55,Closure Permanent


Inspeccionemos un poco la tabla resultante:

In [ ]:
print(df.shape)
df.dtypes

(36, 7)


,0
Notice Date,object
Effective,object
Received,object
Company,object
City,object
No. Of,object
Layoff/Closure,object


🤔 **Preguntas:**

1. ¿Cuántas filas y columnas tiene el `DataFrame`? ¿Coincide con lo que esperabas a partir de `tabla_cruda`?



**Respuesta (Martin):** El `DataFrame` tiene 36 filas y 7 columnas — `df.shape` da `(36, 7)`. Sí coincide con lo esperado: `tabla_cruda` tenía 37 filas en total, pero la primera es el encabezado (`encabezados = tabla_cruda[0]`) y se descuenta al armar `filas_datos = tabla_cruda[1:]`, quedando 36 filas de datos.

2. Mirá el resultado de `df.dtypes`. ¿De qué tipo son todas las columnas? ¿Por qué creés que ocurre esto, incluso en columnas que a simple vista parecen numéricas (como la columna con la cantidad de personas despedidas)?


**Respuesta (Martin):** Todas las columnas son de tipo `object` (texto/string en pandas), incluida `No. Of`, que a simple vista muestra solo números. Esto pasa porque `extract_table()` de `pdfplumber` devuelve **todo** como strings de Python (no intenta inferir tipos), y al construir el `DataFrame` a partir de esa lista de listas, pandas respeta el tipo que recibe: si todos los valores de una columna son `str`, la columna queda como `object`, aunque el contenido "parezca" numérico. Para tener números reales hay que convertir explícitamente con `pd.to_numeric()`, como se pide en la siguiente consigna.

3. Elegí la columna con la cantidad de personas afectadas por cada despido/cierre y convertila a tipo numérico usando `pd.to_numeric()`. ¿Tuviste que limpiar el texto antes de convertirla (por ejemplo, sacar espacios)?

In [ ]:
# 🚀 Tu turno: convertí la columna de cantidad de personas afectadas a numérica

# Martin: la columna con la cantidad de personas afectadas es "No. Of".
# Aplico .str.strip() por prudencia (por si viene con espacios extra en otras páginas),
# y despues pd.to_numeric() para convertirla realmente a números.
df["No. Of"] = pd.to_numeric(df["No. Of"].str.strip())

print("Martin - dtype de 'No. Of' despues de convertir:", df["No. Of"].dtype)
df[["Company", "City", "No. Of"]].head()

Martin - dtype de 'No. Of' despues de convertir: int64


,Company,City,No. Of
0,Maxim Integrated Product,San Jose,150
1,McGraw-Hill Education,Monterey,137
2,Long Beach Memorial Medical Center,Long Beach,90
3,Leidos,El Segundo,72
4,"Bosch Healthcare Systems, Inc.",Palo Alto,55


### 🚀 Desafío final: extraer la tabla de **todas** las páginas

El reporte tiene 16 páginas, y cada una contiene una porción de la misma tabla. Tu desafío es armar **un único `DataFrame`** con todos los registros del documento completo.

Pistas:

- Vas a tener que recorrer `pdf_plumber.pages` con un `for`, igual que en la sección 7.
- Usá `pagina.extract_table()` en cada página.
- Ojo: probablemente solo la **primera página** tenga la fila de encabezados repetida; en las demás páginas la tabla puede empezar directamente con datos.
- Para juntar varios `DataFrame` en uno solo podés usar `pd.concat([...], ignore_index=True)`.
- Al final, guardá el resultado en un archivo CSV con `df_completo.to_csv("warn_report.csv", index=False)`.


In [ ]:
# Tu turno: arma un DataFrame con la tabla completa del documento (las 16 páginas)

# Martin: recorro las 16 páginas y voy armando una lista de DataFrames.
# Ojo (Martin): al inspeccionar página por página, encontré que la ÚLTIMA página (índice 15)
# no es una continuación de la tabla de detalle: trae una tabla RESUMEN con 9 columnas
# (totales por mes/categoría), no las 7 columnas de siempre. La excluyo explícitamente
# comparando la cantidad de columnas contra el encabezado de referencia.

encabezados_ref = None
dfs = []
paginas_omitidas = []

for i, pagina in enumerate(pdf_plumber.pages):
    tabla = pagina.extract_table()
    if not tabla:
        continue

    if i == 0:
        encabezados_ref = tabla[0]
        filas = tabla[1:]
    else:
        # Martin: solo la primera página trae la fila de encabezados repetida
        filas = tabla[1:] if tabla[0] == encabezados_ref else tabla

    if any(len(fila) != len(encabezados_ref) for fila in filas):
        # Martin: página con estructura distinta (la tabla resumen final) -> se omite
        paginas_omitidas.append(i)
        continue

    dfs.append(pd.DataFrame(filas, columns=encabezados_ref))

df_completo = pd.concat(dfs, ignore_index=True)

print("Martin - páginas con tabla distinta y por eso omitidas:", paginas_omitidas)
print("Martin - forma final del DataFrame completo:", df_completo.shape)

df_completo.to_csv("warn_report.csv", index=False)
print("Martin - CSV guardado como warn_report.csv")
df_completo.head()

Martin - páginas con tabla distinta y por eso omitidas: [15]
Martin - forma final del DataFrame completo: (633, 7)
Martin - CSV guardado como warn_report.csv


,Notice Date,Effective,Received,Company,City,No. Of,Layoff/Closure
0,06/22/2015,0 3 / 2 5 / 2 0 16,0 7 / 0 1 / 2 0 15,Maxim Integrated Product,San Jose,150,Closure Permanent
1,06/30/2015,0 8 / 2 9 / 2 0 15,0 7 / 0 1 / 2 0 15,McGraw-Hill Education,Monterey,137,Layoff Unknown at this time
2,06/30/2015,0 8 / 3 0 / 2 0 15,0 7 / 0 1 / 2 0 15,Long Beach Memorial Medical Center,Long Beach,90,Layoff Permanent
3,07/01/2015,0 9 / 0 2 / 2 0 15,0 7 / 0 1 / 2 0 15,Leidos,El Segundo,72,Layoff Permanent
4,07/01/2015,0 9 / 3 0 / 2 0 16,0 7 / 0 1 / 2 0 15,"Bosch Healthcare Systems, Inc.",Palo Alto,55,Closure Permanent


---
## Cierre

No te olvides de **cerrar** los documentos que abriste, para liberar los recursos del sistema (esto es una buena práctica, especialmente cuando trabajás con muchos archivos):


In [ ]:
pdf_plumber.close()
doc.close()
doc_modificado.close()

print("Documentos cerrados correctamente ✅")

Documentos cerrados correctamente ✅


### ✅ Repaso de lo aprendido

En este taller aprendiste a:

- Instalar y cargar `pdfplumber` y `pymupdf`.
- Descargar un PDF público desde internet con `requests`.
- Abrir/importar un PDF con ambas librerías y comparar sus objetos.
- Leer la metadata de un documento.
- Modificar la metadata y guardar un nuevo archivo PDF.
- Acceder a una página específica y extraer su contenido.
- Extraer todo el texto de un documento recorriendo sus páginas.
- Extraer una tabla y convertirla en un `DataFrame` de `pandas`.

### 🔭 Para seguir explorando (fuera de este taller)

- **PDFs escaneados y OCR**: cuando el "texto" de un PDF es en realidad una imagen, ninguna de estas librerías puede extraerlo directamente; hace falta OCR (por ejemplo, con `pytesseract`). Es un tema para otro taller.
- Extracción y manipulación de **imágenes** embebidas en un PDF (`PyMuPDF` tiene herramientas muy potentes para esto).
- **Generación** de PDFs desde cero (por ejemplo, con `reportlab` o `fpdf2`).
- Configuración fina de la detección de tablas en `pdfplumber` (parámetro `table_settings` de `extract_table()`), útil cuando la tabla no se detecta bien de forma automática.


**Todos los bloques agregados están identificados con "[Martin]" en el título de la celda de markdown y en el comentario de la primera línea del código, para diferenciarlos del trabajo original del notebook.**